# 01. Connect walking video to trustworthy skeletons

The model never sees a diagnosis. It sees coordinates estimated from frames. This lesson makes that conversion visible, preserves frame numbering, and explains why a pose cache is only useful when its provenance and missingness are explicit.

## Run locally

Run this notebook independently from the experiment directory. No previous notebook execution is required. The environment installs this local `worldmodels` package and the repository's editable Ambient package together.

```bash
cd experiments/aaai-worldmodels
uv sync
uv run python -m ipykernel install --user --name medwm-gait --display-name "Med-WM gait (Python 3.12)"
uv run jupyter lab
```

Select **Med-WM gait (Python 3.12)**. The project requires Python 3.12. Place local settings in this project's ignored `.env`. The requested `load_env(find_env())` spelling uses aliases for `python-dotenv`; it resolves this experiment's `.env` explicitly and never displays credentials. These lessons need no paid model API.

Set `WM_DATA_MODE=real` to use only actual prepared GAVD skeletons and real MediaPipe/PyTorch/classifier calls. Real mode requires the immutable person-gated clean cohort and fails if those inputs are absent; it never substitutes synthetic data. It embeds a short local walking excerpt and persists numerical outputs and trained lesson checkpoints under a separately named real execution. Run all real lessons with `uv run python scripts/execute_notebooks.py --data-mode real`.

The default `synthetic` mode is an optional fast teaching fixture. Its motions are not medical simulations and cannot establish clinical accuracy. Real lesson training is also a bounded tutorial exercise, distinct from the primary controlled study. Keep all three evidence types separate.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown, SVG
from worldmodels.settings import PROJECT, REPOSITORY, load_env, find_env
load_env(find_env())
torch.set_num_threads(2)
from worldmodels.training import seed_everything
from worldmodels.notebook_support import (data_mode, lesson_data, real_inputs, real_example,
    bounded_preview, lesson_directory, persist_lesson, prepared_path, comparison_path)
REAL = data_mode() == "real"
RELEASE_TEST_RESULTS = os.getenv("WM_RELEASE_TEST_RESULTS", "0") == "1"
EVIDENCE_LABEL = "Real GAVD" if REAL else "Synthetic teaching"
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
seed_everything(27)
print("Data mode:", data_mode(), "| Prepared source:", prepared_path() if REAL else "synthetic fixture")
print("Project:", PROJECT.name, "| Python kernel ready")

### Walking-video context

Real mode embeds only a short original walking excerpt from an accepted normal training sequence. The preview is saved under `data/videos-gavd/previews/`, with its source hash and frame range. It provides context for this lesson without duplicating the full original. Synthetic mode enables the same real viewing aid only when `WM_EMBED_VIDEO=1`.

In [ ]:
if REAL or os.getenv("WM_EMBED_VIDEO", "0") == "1":
    from IPython.display import Video
    preview_path = bounded_preview()
    display(Video(str(preview_path), embed=True, html_attributes='controls preload="metadata"'))
    print("Bounded source excerpt:", preview_path.relative_to(REPOSITORY))
else:
    print("Synthetic mode: set WM_EMBED_VIDEO=1 for a real-video context excerpt.")

## Step 1. Find a real source without duplicating media

Existing originals live under `data/videos-gavd/<label>/`. The resolver also supports `sources/` for new downloads, so each YouTube ID can be stored once. New previews belong under `data/videos-gavd/previews/`. We preserve existing media and never copy a full video into multiple condition folders for this experiment.

In [ ]:
from worldmodels.data import build_inventory, find_video
inventory, audit = build_inventory(REPOSITORY)
example = real_example() if REAL else inventory.loc[inventory.eligible & (inventory.label == "normal")].sort_values("sequence_id").iloc[0]
video_path = find_video(REPOSITORY, example.source_id, example.label)
annotation = pd.read_csv(REPOSITORY / example.csv_path, keep_default_na=False)
display(annotation[["seq", "frame_num", "cam_view", "gait_pat"]].head())
print("Local source:", video_path.relative_to(REPOSITORY))

## Step 2. Watch the annotated interval

Set `WM_EMBED_VIDEO=1` in `.env` to embed this local original. Embedded video makes a notebook larger; clear its output before sharing the notebook. Playback starts near the annotated interval, which is more useful than watching an unrelated introduction. Real mode always embeds the bounded excerpt. The original can also be opened in your local video player.

Check that the visible person matches the intended sequence, that walking is present, and that there are no cuts across the requested interval. A skeleton detector can choose the wrong person in a multi-person scene. The current adapter selects the first full-image pose, so pilot outputs remain unverified for person identity until reviewed.

In [ ]:
import cv2
from worldmodels.pose import video_preview
cap = cv2.VideoCapture(str(video_path))
source_fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
start_seconds = (int(example.first_frame) - 1) / source_fps
end_seconds = int(example.last_frame) / source_fps
if REAL or os.getenv("WM_EMBED_VIDEO", "0") == "1":
    from IPython.display import Video
    display(Video(str(bounded_preview(example)), embed=True))
else:
    print(f"Open {video_path.name} near {start_seconds:.2f} to {end_seconds:.2f} seconds; set WM_EMBED_VIDEO=1 for inline playback.")

## Step 3. Preserve the frame boundary exactly once

GAVD frame 1 means the first source frame. OpenCV's decoded index 0 means the same frame. The extraction adapter converts `frame_num - 1` once and then passes the decoded RGB image to Ambient. It does not pass a second shifted index to another frame-extraction API.

Bounding boxes were annotated against an earlier image representation. They should not be applied blindly when the downloaded image differs in scale, borders, or framing. The current pilot uses full images and records that limitation.

In [ ]:
mapping = annotation[["frame_num"]].head(8).copy()
mapping["decoded_index"] = mapping.frame_num - 1
mapping["nominal_seconds"] = mapping.decoded_index / source_fps
display(mapping)
assert (mapping.decoded_index >= 0).all()

## Step 4. Run a bounded real extraction

Real mode always executes MediaPipe on 16 sampled frames in a fresh lesson-specific cache. In synthetic mode, set `WM_RUN_POSE=1` to request the same actual extraction. It uses the local MediaPipe model and Ambient's public `SequenceKeypointExtractor.extract_from_image`. The cache records model, video, annotation, extractor, and adapter fingerprints. Failed detections retain their frame positions with zero confidence and missing coordinates. They are not removed and mistaken for adjacent time steps.

This small extraction checks the pipeline. It is not enough to characterize a full gait cycle or establish performance. The full preparation command in the capstone can process the selected cohort.

In [ ]:
from worldmodels.pose import extract_sequence, make_windows
pose = None
if REAL or os.getenv("WM_RUN_POSE", "0") == "1":
    model_path = Path(os.getenv("WM_POSE_MODEL") or REPOSITORY / "data/models/pose_landmarker_lite.task")
    pose = extract_sequence(REPOSITORY / example.csv_path, REPOSITORY, model_path,
                            (lesson_directory("01_video_pose_and_quality") / "poses") if REAL else PROJECT / "artifacts/poses", frame_step=2, max_frames=16)
    print("Pixels:", pose["xy"].shape, "| confidence:", pose["confidence"].shape)
    print("Observed frame numbers:", pose["frame_numbers"])
    x, valid, metadata = make_windows(pose)
    print("Accepted windows:", len(x))
else:
    print("Real extraction is available with WM_RUN_POSE=1. No pose result is fabricated.")

## Step 5. Understand normalization and missingness

The 16 selected landmarks are centered on the pelvis. The median observed torso length in the first eight frames supplies a single scale. Future frames cannot change this scale. Missing pelvis observations invalidate the affected frame. This representation describes relative articulation rather than global travel through the scene.

Random completion has a separate caveat: normalization is performed before the random mask is drawn, so hidden hips or shoulders may have contributed to the transform. The task is completion of a pre-normalized articulation sequence. See the joint contract before making stronger claims about raw-coordinate conditional prediction.

In [ ]:
from worldmodels.notebook_support import lesson_data
fixture = lesson_data(n=2)
x_demo = fixture["x"][0]
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(x_demo[0,:,0], x_demo[0,:,1], c=np.arange(16), cmap="viridis")
ax.invert_yaxis(); ax.set_aspect("equal")
ax.set(xlabel="Relative x in torso units", ylabel="Relative y in torso units", title=EVIDENCE_LABEL + " skeleton, first frame")
plt.show()

## Step 6. Audit what did not survive

The fresh inventory excludes known missing videos, an insufficient source range, and four sequences with a documented video-reference mismatch. Pose processing creates additional attrition. Keep its reasons and count retained sources by label and partition. Confidence thresholds are study settings, not universal Ambient defaults.

The tutorials use a confidence cutoff of 0.5 and minimum observed fraction of 0.7. A sensitivity analysis should vary these using development data and report how many sources disappear. Do not select a quality cutoff because it makes the final test score look better.

In [ ]:
display(inventory.loc[~inventory.eligible].groupby(["label", "exclusion_reason"]).size().rename("sequences").reset_index())
print("Pose filtering has not been inferred from media availability.")

In [ ]:
lesson_evidence = persist_lesson("01_video_pose_and_quality", globals())
print("Evidence mode:", lesson_evidence.get("mode"))

## What to retain

Record the data identity, settings, executed commands, and limitations with any result. A successful cell execution is implementation evidence, not proof that the method generalizes. See the [data protocol](docs/data_protocol.md), [method review](docs/literature.md), and [README](README.md) before changing the study.